### Import modules and utility code

In [ ]:
import os
import numpy as np
from score_config import ScoreConfig
from score_config_utils import ureg
print(os.getcwd())

### Create a configuration file for SCORE following the NXmicrostructure_score_config.nxdl.xml contributed application definition

<div class="alert alert-block alert-info">Modify those values in this populated template that match your case.</div>

In [ ]:
SIMID = 666

MY_MATERIAL = [
    ("shear_modulus", ureg.Quantity(np.float64(9.5e9), ureg.pascal)),
    ("burgers_vector", ureg.Quantity(np.float64(0.286), ureg.nanometer)),
    ("melting_temperature", ureg.Quantity(np.float64(800.), ureg.degree).to(ureg.kelvin)),
]

MY_USER = ["Markus Kühbach", "Vitesh Shah", "Markus Kühbach", "Sebastian Samberger", "Georg Falkinger", "Franz Roters"]

MY_INIT_DAMASK = ("file_name", "Tphase_Vf_01_tensionX_remeshed_increment19000_segmented.dream3D")

MY_MOBILITY_RH = [
    ("m_null",ureg.Quantity(np.float64(300.), ureg.meter ** 4 / (ureg.joule * ureg.second))),
    ("enthalpy", ureg.Quantity(np.float64(2.0314), ureg.eV)),
    # ("c1", ureg.Quantity(np.float64(0.9))),
    # ("c2", ureg.Quantity(np.float64(5.))),
    # ("c3", ureg.Quantity(np.float64(9.))),
]

MY_TIME_TEMPERATURE = [
    ("time", ureg.Quantity(np.asarray(
        [0., 5.8, 100.8, 103.7, 105., 200.8, 205.8, 208.7, 300., 305.8, 405.8, 408.7, 600.], np.float64), ureg.second)),
    ("temperature", ureg.Quantity(np.asarray(
        [20., 500., 500., 20., 20., 500., 500., 20., 20., 500., 500., 20., 20.], np.float64), ureg.degC).to(ureg.kelvin)),
]

MY_GRID_CUBOIDAL = [
    ("extent", ureg.Quantity(np.asarray([19, 15, 11], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.5), ureg.micrometer))
]

MY_GRID_DAMASK = [
    ("extent", ureg.Quantity(np.asarray([128, 52, 64], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.4982421), ureg.micrometer)),
]

MY_SIM_CONTROL = [
    ("max_x", ureg.Quantity(np.float64(1.))),
    ("max_time", ureg.Quantity(np.float64(3600.), ureg.second)),
    ("max_iteration", ureg.Quantity(np.uint32(10000))),
    ("max_delta_x", ureg.Quantity(np.float64(0.1))),
    ("x_set", ureg.Quantity(np.asarray(np.linspace(0., stop=1., num=101, endpoint=True), np.float64))),
]

In [ ]:
print("Generating NeXus/HDF5 configuration file for examples/verification")
nxs = ScoreConfig(0)
nxs.write_config_hfive_file()
del nxs

In [ ]:
print("Generating NeXus/HDF5 configuration file for examples/damask")
nxs = ScoreConfig(1)
nxs.user(MY_USER)
nxs.sample("Al")
nxs.material(MY_MATERIAL)
nxs.deformation(model="damask", parameters=MY_INIT_DAMASK)
nxs.nucleation(model="damask")
nxs.mobility(model="rollett_holm", parameters=MY_MOBILITY_RH)
nxs.time_temperature(MY_TIME_TEMPERATURE)
nxs.discretization(MY_GRID_DAMASK)
nxs.numerics(MY_SIM_CONTROL)
nxs.write_config_hfive_file()
del nxs

In [ ]:
class ScoreConfig(ConfigBase):
    # attributes use @ as a marker/escape character
    # groups are all those concepts which have an attribute @NX_class
    # fields are all those other concepts with an attribute %value
    def __init__(self, simid):
        self.deg2kelvin = 273.15
        self.sim_id = simid
        self.entry_id = 1
        self.yaml_file_name = f"NXmicrostructure_score_config.SimID.{self.sim_id}.yaml"
        self.hdf5_file_name = f"SCORE.Config.SimID.{self.sim_id}.nxs"
        self.cfg: Dict[str, Any] = {}
        self.cfg["@NX_class"] = "NXroot"
        self.cfg["@default"] = f"entry{self.entry_id}"
        self.cfg[f"entry{self.entry_id}/@NX_class"] = "NXentry"
        self.cfg[f"entry{self.entry_id}/@default"] = "time_temperature"
        self.cfg[f"entry{self.entry_id}/definition"] = "NXmicrostructure_score_config"
        self.cfg[f"entry{self.entry_id}/definition/@version"] = f"{get_nexus_version()}"
        # "simulation_identifier": None,
        # "start_time": None,
        # "end_time": None,
        # "program1": None
        # physics
        self.description()
        self.user()
        self.sample()
        self.material()
        self.deformation()
        self.nucleation()
        self.mobility()
        self.recovery()
        self.drag()
        self.component_analysis()
        self.time_temperature()
        # numerics
        self.discretization()
        self.numerics()
        self.solitary_unit_model()

    def description(self):
        """Configure ELN-like description."""
        trg = f"entry{self.entry_id}/"
        self.cfg[f"{trg}description"] = "Example showing how to bring recrystallization simulations with SCORE into NOMAD."""
    
    def user(self):
        """Configure the user."""
        for idx, name in enumerate(["MarkusK", "ViteshS"]):
            trg = f"entry{self.entry_id}/user{idx + 1}/"
            self.cfg[f"{trg}@NX_class"] = "NXuser"
            self.cfg[f"{trg}name"] = name

    def sample(self):
        """Configure contextualizing details about the sample."""
        trg = f"entry{self.entry_id}/sample/"
        self.cfg[f"{trg}@NX_class"] = "NXsample"
        self.cfg[f"{trg}dimensionality"] = ureg.Quantity(np.uint32(3))
        self.cfg[f"{trg}type"] = "simulation"
        self.cfg[f"{trg}atom_types"] = "Al"        
    
    def material(self):
        """Configure material model single-phase only."""
        trg = f"entry{self.entry_id}/material/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}shear_modulus"] = ureg.Quantity(np.float64(9.5e9), ureg.pascal)
        self.cfg[f"{trg}burgers_vector"] = ureg.Quantity(np.float64(0.286), ureg.nanometer)
        self.cfg[f"{trg}melting_temperature"] = ureg.Quantity(np.float64(800. + self.deg2kelvin), ureg.kelvin)

    def deformation(self):
        """Configure deformed (initial) microstructure."""
        trg = f"entry{self.entry_id}/deformation/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}model"] = "damask"
        self.cfg[f"{trg}damask/@NX_class"] = "NXnote"
        damask_file_name = "Tphase_Vf_01_tensionX_remeshed_increment19000_segmented.dream3D"
        self.cfg[f"{trg}damask/file_name"] = f"{damask_file_name}"  # how the score application sees this file
        # given that the application is in score_root/build but the examples in score_root/examples/damask
        self.cfg[f"{trg}damask/algorithm"] = "sha256"
        # from the perspective of this JupyterNotebook's current working directory (e.g. check os.getcwd())
        self.cfg[f"{trg}damask/checksum"] = get_sha256_of_file_content(f"../../examples/damask/{damask_file_name}")

    def nucleation(self):
        """Configure nucleation model."""
        trg = f"entry{self.entry_id}/nucleation/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}spatial_distribution"] = "damask"
        self.cfg[f"{trg}number_of_nuclei"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}incubation_time"] = "site_saturation"
        # self.cfg[f"{trg}incubation_time_sigma"] = ureg.Quantity(np.float64(1000.), ureg.second)
        self.cfg[f"{trg}orientation"] = "damask"
        self.cfg[f"{trg}ensemble/@NX_class"] = "NXparameters"
        self.cfg[f"{trg}ensemble/bunge_euler"] = ureg.Quantity(np.asarray(
            [[0., 30., 45.]], np.float64) / 180. * np.pi, ureg.radian)
        self.cfg[f"{trg}ensemble/incubation_time"] = ureg.Quantity(np.asarray(
            [0.], np.float64), ureg.second)

    def mobility(self):
        """Configure grain boundary mobility model."""
        trg = f"entry{self.entry_id}/grain_boundary_mobility/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        model = "rollett_holm"  # "sebald_gottstein"
        if model == "sebald_gottstein":
            self.cfg[f"{trg}@NX_class"] = "NXparameters"
            self.cfg[f"{trg}model"] = model
            trg += "sebald_gottstein/"
            self.cfg[f"{trg}@NX_class"] = "NXparameters"
            self.cfg[f"{trg}lagb_to_hagb_cut"] = ureg.Quantity(np.float64(15.) / 180. * np.pi, ureg.radian)
            self.cfg[f"{trg}lagb_pre_factor"] = ureg.Quantity(np.float64(30.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}lagb_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            self.cfg[f"{trg}hagb_pre_factor"] = ureg.Quantity(np.float64(30.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}hagb_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            self.cfg[f"{trg}special_disorientation"] = ureg.Quantity(np.float64(10.) / 180. * np.pi, ureg.radian)
            self.cfg[f"{trg}special_pre_factor"] = ureg.Quantity(np.float64(30.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}special_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            # MAXDISORI_TO_40DEG111
        elif model == "rollett_holm":
            self.cfg[f"{trg}model"] = model
            trg += "rollett_holm/"
            self.cfg[f"{trg}@NX_class"] = "NXparameters"
            self.cfg[f"{trg}m_null"] = ureg.Quantity(np.float64(300.), ureg.meter ** 4 / (ureg.joule * ureg.second))
            self.cfg[f"{trg}enthalpy"] = ureg.Quantity(np.float64(2.0314), ureg.eV)
            self.cfg[f"{trg}c1"] = ureg.Quantity(np.float64(0.9))
            self.cfg[f"{trg}c2"] = ureg.Quantity(np.float64(5.))
            self.cfg[f"{trg}c3"] = ureg.Quantity(np.float64(9.))

    def recovery(self):
        """Configure stored elastic energy recovery model."""
        trg = f"entry{self.entry_id}/stored_energy_recovery/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}model"] = "none"

    def drag(self):
        """Configure reduction of grain boundary mobility by dispersoids (Zener-Smith drag)."""
        trg = f"entry{self.entry_id}/dispersoid_drag/"
        self.cfg[f"{trg}@NX_class"] = "NXdata"
        self.cfg[f"{trg}model"] = "none"
        # trg = "entry{self.entry_id}/dispersoid_drag/zener_smith/"
        # self.cfg[f"{trg}@NX_class"] = "NXparameters"
        # self.cfg[f"{trg}@signal"] = "radius"
        # self.cfg[f"{trg}@axes"] = "time"
        # self.cfg[f"{trg}@time_indices"] = ureg.Quantity(np.uint32(0))
        # self.cfg[f"{trg}@radius_indices"] = ureg.Quantity(np.uint32(1))
        # self.cfg[f"{trg}title"] = "r(t)"  
        # self.cfg[f"{trg}pre_factor"] = ureg.Quantity(np.float64(1.5))
        # self.cfg[f"{trg}surface_energy"] = ureg.Quantity(np.float64(0.324), ureg.joule / (ureg.meter ** 2))
        # self.cfg[f"{trg}time/@long_name"] = "t (s)"
        # self.cfg[f"{trg}radius/@long_name"] = "r (nm)"
        # self.cfg[f"{trg}time"] = ureg.Quantity(np.asarray([0., 3600.], np.float64), ureg.second)
        # self.cfg[f"{trg}radius"] = ureg.Quantity(np.asarray([0., 0.], np.float64), ureg.nanometer)

    def component_analysis(self):
        """Configure which macrotexture components the code should bookkeep for volume evolution."""
        trg = f"entry{self.entry_id}/component_analysis/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}name"] = ["Alpha001-110", "Alpha112-110"]
        self.cfg[f"{trg}bunge_euler"] = ureg.Quantity(
            np.asarray([[0., 0., 0.],
                        [0., 35., 45.]], np.float64) / 180. * np.pi, ureg.radian)
        self.cfg[f"{trg}theta"] = ureg.Quantity(
            np.asarray([10., 10.], np.float64) / 180. * np.pi, ureg.radian)

    def time_temperature(self):
        """Configure the simulated annealing schedule."""
        trg = f"entry{self.entry_id}/time_temperature/"
        self.cfg[f"{trg}@NX_class"] = "NXdata"
        self.cfg[f"{trg}@signal"] = "temperature"
        self.cfg[f"{trg}@axes"] = "time"
        self.cfg[f"{trg}@time_indices"] = ureg.Quantity(np.uint32(0))
        self.cfg[f"{trg}@temperature_indices"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}title"] = "T(t)"        
        self.cfg[f"{trg}time/@long_name"] = "t (s)"
        self.cfg[f"{trg}temperature/@long_name"] = "T (K)"
        self.cfg[f"{trg}time"] = ureg.Quantity(np.asarray(
            [0., 5.8, 100.8, 103.7, 105., 200.8, 205.8, 208.7, 300., 305.8, 405.8, 408.7, 600.], np.float64), ureg.second)
        self.cfg[f"{trg}temperature"] = ureg.Quantity(np.asarray(
            [20., 500., 500., 20., 20., 500., 500., 20., 20., 500., 500., 20., 20.], np.float64) + self.deg2kelvin, ureg.kelvin)
    
    def discretization(self):
        """Configure microstructure instantiation."""
        trg = f"entry{self.entry_id}/discretization/"
        self.cfg[f"{trg}@NX_class"] = "NXmicrostructure"
        trg += "grid/"
        self.cfg[f"{trg}@NX_class"] = "NXcg_grid"
        self.cfg[f"{trg}extent"] = ureg.Quantity(np.asarray([128, 52, 64], np.uint32))
        self.cfg[f"{trg}cell_dimensions"] = ureg.Quantity(np.float64(0.4982421), ureg.micrometer)
       
    def numerics(self):
        """Configure stop criteria and numerics."""
        trg = f"entry{self.entry_id}/numerics/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}max_x"] = ureg.Quantity(np.float64(1.))
        self.cfg[f"{trg}max_time"] = ureg.Quantity(np.float64(3600.), ureg.second)
        self.cfg[f"{trg}max_iteration"] = ureg.Quantity(np.uint32(10000))
        self.cfg[f"{trg}max_delta_x"] = ureg.Quantity(np.float64(0.1))
        self.cfg[f"{trg}x_set"] = ureg.Quantity(np.asarray(np.linspace(0., stop=1., num=101, endpoint=True), np.float64))
        # self.cfg[f"{trg}percolation"] = False
        self.cfg[f"{trg}cell_cache/@NX_class"] = "NXparameters"
        self.cfg[f"{trg}cell_cache/initial"] = ureg.Quantity(np.float64(0.2))
        self.cfg[f"{trg}cell_cache/realloc"] = ureg.Quantity(np.float64(1.))
        self.cfg[f"{trg}cell_cache/defragment"] = True
        self.cfg[f"{trg}cell_cache/defragment_x"] = ureg.Quantity(np.asarray([0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
                      0.92, 0.94, 0.96, 0.98, 0.99, 0.999, 0.9999], np.float64))

    def solitary_unit_model(self):
        """Configure solitary unit post-processing approach."""
        trg = f"entry{self.entry_id}/solitary_unit/"
        self.cfg[f"{trg}@NX_class"] = "NXparameters"
        self.cfg[f"{trg}apply"] = False
        self.cfg[f"{trg}number_of_domains"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}rediscretization"] = ureg.Quantity(np.uint32(10000))
        # self.cfg[f"{trg}domain_identifier"] = None